# AASIST-L — ASVspoof 2019 LA (A0), Kaggle 2×T4

Chạy bằng **Save Version → Save & Run All** (không chạy tay từng cell). Xem `kaggle/README.md`.

- Cell 0: tham số (`TRIAL_RUN = True` cho lần chạy thử: 10 phút, tối đa 2 epoch). Cell 1: cài đặt. Cell 2: **chấm pretrained AASIST-L (chế độ cửa sổ)** → kiểm chứng pipeline.
- Cell 3–5: train 2 seed song song (mỗi seed 1 GPU), `--max_minutes`, resume tự động; xuất file điểm khi run đã xong.
- Resume: phiên sau gắn **output của version trước** làm input và đặt `PREV_RUN`.

Quy ước: nhãn 1 = thật, 0 = giả; điểm cao = thật; CE [0.1, 0.9]; chọn checkpoint theo dev EER (cửa sổ).

In [ ]:
# ===== Cell 0 — THAM SỐ (chỉ sửa ở đây) =====
REPO_URL = "https://github.com/Phvster/antispoof-benchmark.git"
REPO_REF = "__REPO_REF__"          # HASH COMMIT cố định (không dùng tên nhánh); env.json ghi git_commit
NPY_DATASET = "/kaggle/input/asvspoof2019-la-npy"   # TÊN DATASET LÀ THAM SỐ — layout xem README
PROTO_DIR = f"{NPY_DATASET}/protocols"              # asvspoof2019_la_{train,dev,eval}.csv
PREV_RUN = None                    # vd "/kaggle/input/aasist-l-run-v1" (output version trước) để resume
SEEDS = [0, 1]                     # 2 seed song song trên 2×T4; seed 2 chạy ở version sau
TRIAL_RUN = True                   # chạy thử: 10 phút, tối đa 2 epoch/phiên. Train thật: False
MAX_MINUTES = 10 if TRIAL_RUN else 660          # ngân sách train mỗi seed trong phiên (Kaggle tối đa 12 h)
STOP_AFTER_EPOCHS = 2 if TRIAL_RUN else None    # tạm dừng sau N epoch của phiên (không đổi lịch cosine)
NUM_WORKERS = 2                    # mỗi tiến trình; 2 tiến trình chia 4 CPU của Kaggle
RUN_PRETRAINED_CHECK = PREV_RUN is None   # chỉ chấm pretrained ở phiên đầu (tiết kiệm thời gian)
EXPORT_FULL = False                # True: xuất thêm chế độ cả file (chậm, batch 1)
TRAIN_NAME, DEV_NAME, EVAL_NAME, AUG = "la19", "la19dev", "la19eval", "A0"

AASIST_L_URL = "https://github.com/clovaai/aasist/raw/a04c9863f63d44471dde8a6abcb3b082b07cd1d1/models/weights/AASIST-L.pth"
AASIST_L_MD5 = "a2f6cecccdcfb78a3eb0273cf9face2c"

WORK = "/kaggle/working"
REPO = "/tmp/antispoof-benchmark"  # NGOÀI /kaggle/working -> repo (và token) không lọt vào output version
RUNS = f"{WORK}/runs"
SCORES = f"{WORK}/scores"

In [ ]:
# ===== Cell 1 — cài đặt =====
import os, subprocess, sys, json, shutil, time, hashlib

def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    return subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

def clone_repo():
    # Repo private: đọc token chỉ-đọc từ Kaggle Secrets (tên GITHUB_TOKEN). Không in, không lưu token.
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("Không đọc được secret GITHUB_TOKEN (repo public thì bỏ qua):", type(e).__name__)
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1) if token else REPO_URL
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone lỗi (kiểm tra secret GITHUB_TOKEN + Add-ons > Secrets đã bật cho notebook):\n" + msg)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)  # bỏ token khỏi .git/config

if not os.path.exists(REPO):
    clone_repo()
sh(["git", "-C", REPO, "checkout", "-q", REPO_REF])
sh(["git", "-C", REPO, "log", "-1", "--oneline"])
sh([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements.txt"])  # KHÔNG cài torch
os.chdir(REPO)
sys.path.insert(0, REPO)

import torch
print("torch", torch.__version__, "| GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for sub in ("train", "dev", "eval"):
    assert os.path.isdir(f"{NPY_DATASET}/{sub}"), f"thiếu {NPY_DATASET}/{sub} — xem kaggle/README.md"
    assert os.path.exists(f"{PROTO_DIR}/asvspoof2019_la_{sub}.csv"), f"thiếu protocol {sub}"
from common.data import md5sum
for sub in ("train", "dev", "eval"):
    print(sub, "protocol md5", md5sum(f"{PROTO_DIR}/asvspoof2019_la_{sub}.csv"))
os.makedirs(RUNS, exist_ok=True); os.makedirs(SCORES, exist_ok=True)

In [ ]:
# ===== Cell 2 — chấm pretrained AASIST-L, chế độ cửa sổ (kiểm chứng pipeline) =====
# Kỳ vọng (clovaai): EER ≈ 0,99%, min t-DCF ≈ 0,0309 (t-DCF chấm bằng eval/score.py của B).
if RUN_PRETRAINED_CHECK:
    import urllib.request
    from common.data import read_protocol
    from common.metrics import eer_from_labels
    w = f"{WORK}/weights/AASIST-L.pth"
    os.makedirs(os.path.dirname(w), exist_ok=True)
    if not os.path.exists(w):
        urllib.request.urlretrieve(AASIST_L_URL, w)
    assert md5sum(w) == AASIST_L_MD5, "md5 AASIST-L.pth sai"
    t0 = time.time()
    sh([sys.executable, "-m", "common.score_export", "--config", "models/aasist_l/config.yaml",
        "--pretrained", w, "--protocol", f"{PROTO_DIR}/asvspoof2019_la_eval.csv",
        "--audio_dir", f"{NPY_DATASET}/eval", "--ext", ".npy", "--mode", "window",
        "--train_name", TRAIN_NAME, "--test_name", EVAL_NAME, "--aug", AUG, "--seed", "pretrained",
        "--out_dir", SCORES, "--batch_size", "64", "--num_workers", "4"])
    path = f"{SCORES}/window/aasist_l_{TRAIN_NAME}_{EVAL_NAME}_{AUG}_seedpretrained.txt"
    df = read_protocol(f"{PROTO_DIR}/asvspoof2019_la_eval.csv")
    score = dict(l.split() for l in open(path))
    s = [float(score[f]) for f in df["file_id"]]
    eer, _ = eer_from_labels(s, df["label"].values)
    print(f"pretrained AASIST-L, 2019 LA eval, cửa sổ: EER = {eer:.3f}%  ({time.time() - t0:.0f}s)")
    if not 0.5 <= eer <= 1.5:
        raise RuntimeError(f"EER {eer:.2f}% lệch xa 0,99% -> pipeline sai (kiểm tra nhãn/dấu điểm). DỪNG.")
else:
    print("Bỏ qua chấm pretrained (phiên resume).")

In [ ]:
# ===== Cell 3 — resume: chép runs/ + scores/ của version trước =====
if PREV_RUN:
    for sub in ("runs", "scores"):
        src = f"{PREV_RUN}/{sub}"
        if os.path.isdir(src):
            shutil.copytree(src, f"{WORK}/{sub}", dirs_exist_ok=True)
            print("chép", src, "->", f"{WORK}/{sub}")
for k in SEEDS:
    st = f"{RUNS}/aasist_l_seed{k}/run_status.json"
    print(k, json.load(open(st)) if os.path.exists(st) else "chưa có run")

In [ ]:
# ===== Cell 4 — train 2 seed song song, mỗi seed một GPU =====
n_gpu = torch.cuda.device_count()
assert n_gpu >= len(SEEDS), f"cần {len(SEEDS)} GPU (Accelerator: GPU T4 x2), có {n_gpu}; hoặc đặt SEEDS=[0]"
procs = []
for gpu, k in enumerate(SEEDS):
    out = f"{RUNS}/aasist_l_seed{k}"
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "-m", "common.train", "--config", "models/aasist_l/config.yaml",
           "--seed", str(k), "--out_dir", out, "--resume", "auto", "--max_minutes", str(MAX_MINUTES),
           "--num_workers", str(NUM_WORKERS), "--ext", ".npy",
           *(["--stop_after_epochs", str(STOP_AFTER_EPOCHS)] if STOP_AFTER_EPOCHS else []),
           "--train_protocol", f"{PROTO_DIR}/asvspoof2019_la_train.csv", "--train_audio_dir", f"{NPY_DATASET}/train",
           "--dev_protocol", f"{PROTO_DIR}/asvspoof2019_la_dev.csv", "--dev_audio_dir", f"{NPY_DATASET}/dev"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    log = open(f"{out}/stdout_{int(time.time())}.log", "w")
    print(f"seed {k} -> GPU {gpu}: {out}")
    procs.append((k, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, cwd=REPO), log))

while any(p.poll() is None for _, p, _ in procs):
    time.sleep(300)
    for k, p, log in procs:
        f = f"{RUNS}/aasist_l_seed{k}/log.csv"
        last = open(f).read().strip().splitlines()[-1] if os.path.exists(f) else "(chưa xong epoch nào)"
        print(time.strftime("%H:%M"), f"seed {k}:", last, flush=True)
for k, p, log in procs:
    log.close()
    print(f"seed {k}: exit {p.returncode}")
    if p.returncode != 0:
        print(open(log.name).read()[-3000:])
assert all(p.returncode == 0 for _, p, _ in procs), "có tiến trình train lỗi — xem log ở trên"

In [ ]:
# ===== Cell 5 — xuất file điểm từ best.pt cho run đã xong (finished / early_stopped) =====
for k in SEEDS:
    out = f"{RUNS}/aasist_l_seed{k}"
    st = json.load(open(f"{out}/run_status.json"))
    print(f"seed {k}:", st)
    if st["state"] not in ("finished", "early_stopped"):
        print("  -> chưa xong (paused): Save Version rồi resume ở version sau.")
        continue
    modes = ["window"] + (["full"] if EXPORT_FULL else [])
    for mode in modes:
        for test_name, sub in ((DEV_NAME, "dev"), (EVAL_NAME, "eval")):
            sh([sys.executable, "-m", "common.score_export", "--checkpoint", f"{out}/best.pt",
                "--protocol", f"{PROTO_DIR}/asvspoof2019_la_{sub}.csv", "--audio_dir", f"{NPY_DATASET}/{sub}",
                "--ext", ".npy", "--mode", mode, "--train_name", TRAIN_NAME, "--test_name", test_name,
                "--aug", AUG, "--seed", str(k), "--out_dir", SCORES, "--batch_size", "64", "--num_workers", "4"])
sh(f"ls -R {SCORES} | head -50")

In [ ]:
# ===== Cell 6 — tóm tắt (output của version này = input resume cho version sau) =====
import csv
for k in SEEDS:
    f = f"{RUNS}/aasist_l_seed{k}/log.csv"
    if os.path.exists(f):
        secs = [float(r["epoch_sec"]) for r in csv.DictReader(open(f))]
        print(f"seed {k}: {len(secs)} epoch, trung bình {sum(secs) / len(secs) / 60:.1f} phút/epoch "
              f"(gồm train + chấm dev 24.844 file)")
for k in SEEDS:
    out = f"{RUNS}/aasist_l_seed{k}"
    print(f"== seed {k}")
    for f in ("run_status.json", "env.json"):
        if os.path.exists(f"{out}/{f}"):
            print(open(f"{out}/{f}").read())
    if os.path.exists(f"{out}/log.csv"):
        print("".join(open(f"{out}/log.csv").readlines()[-5:]))
shutil.rmtree(f"{WORK}/weights", ignore_errors=True)   # không giữ checkpoint pretrained trong output